# AI-Powered Risk Early Warning System - Demo

This notebook demonstrates the complete workflow of the Risk Early Warning System.

## 1. Setup and Imports

In [3]:
import sys
sys.path.append('../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import our modules
from app.services.data_processing import DataPipeline, FeatureEngineer
from app.ml.risk_models import (
    DelayRiskPredictor,
    CostOverrunPredictor,
    ResourceBottleneckDetector,
    RiskScoreCalculator
)
from app.services.sample_data_generator import SampleDataGenerator

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print("✅ Setup complete!")

✅ Setup complete!


## 2. Generate Sample Data

In [4]:
# Generate sample data
generator = SampleDataGenerator(seed=42)
result = generator.generate_all()

print(f"\n✅ Generated {result['projects']} projects and {result['sprints']} sprints")
print(f"📁 Data saved to: {result['output_directory']}")

Generating Sample Data for IT Analytics Platform
Generated 10 projects
Generated 60 sprints
Generated 500 tasks
Generated 130 metric records
Generated 240 resource utilization records
Generated 100 Kaizen logs
All sample data generated in: data/raw/sample

✅ Generated 10 projects and 60 sprints
📁 Data saved to: data/raw/sample


## 3. Load and Explore Data

In [5]:
# Load the generated data
data_dir = '../data/raw/sample'

projects_df = pd.read_csv(f'{data_dir}/projects.csv')
metrics_df = pd.read_csv(f'{data_dir}/project_metrics.csv')
resource_df = pd.read_csv(f'{data_dir}/resource_utilization.csv')
tasks_df = pd.read_csv(f'{data_dir}/tasks.csv')

print("📊 Data Loaded Successfully")
print(f"\nProjects: {len(projects_df)}")
print(f"Metrics Records: {len(metrics_df)}")
print(f"Resource Records: {len(resource_df)}")
print(f"Tasks: {len(tasks_df)}")

# Display sample
print("\n📋 Sample Project Data:")
projects_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/sample/projects.csv'

## 4. Data Processing & Feature Engineering

In [ ]:
# Initialize data pipeline
pipeline = DataPipeline()

# Convert date columns
metrics_df['metric_date'] = pd.to_datetime(metrics_df['metric_date'])

# Process metrics data
processed_metrics = pipeline.process_project_data(metrics_df)

# Process resource data
processed_resources = pipeline.process_resource_data(resource_df)

print("✅ Data processing complete")
print(f"\nProcessed metrics shape: {processed_metrics.shape}")
print(f"Processed resources shape: {processed_resources.shape}")

# Display processed data
processed_metrics.head()

## 5. Exploratory Data Analysis

In [ ]:
# Visualize velocity trends
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Velocity trend
for project_id in processed_metrics['project_id'].unique()[:5]:
    project_data = processed_metrics[processed_metrics['project_id'] == project_id]
    axes[0, 0].plot(project_data['metric_date'], project_data['sprint_velocity'], 
                   label=f'Project {project_id}', marker='o')
axes[0, 0].set_title('Sprint Velocity Trends')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Velocity (Points/Day)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Completion rate distribution
axes[0, 1].hist(processed_metrics['completion_rate'], bins=20, edgecolor='black')
axes[0, 1].set_title('Completion Rate Distribution')
axes[0, 1].set_xlabel('Completion Rate')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Bug reopen rate
axes[1, 0].scatter(processed_metrics['total_bugs'], 
                  processed_metrics['bug_reopen_rate'],
                  alpha=0.6, s=50)
axes[1, 0].set_title('Bugs vs Reopen Rate')
axes[1, 0].set_xlabel('Total Bugs')
axes[1, 0].set_ylabel('Reopen Rate')
axes[1, 0].grid(True, alpha=0.3)

# Resource utilization
axes[1, 1].boxplot([processed_resources['utilization_rate']])
axes[1, 1].set_title('Resource Utilization Distribution')
axes[1, 1].set_ylabel('Utilization Rate')
axes[1, 1].axhline(y=1.0, color='r', linestyle='--', label='100% Utilization')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 EDA Visualizations complete")

## 6. Train ML Models

In [ ]:
# Prepare training data
train_data = processed_metrics.copy()

# Delay Risk Model
print("🤖 Training Delay Risk Predictor...")
delay_predictor = DelayRiskPredictor('xgboost')

X_delay = delay_predictor.prepare_features(train_data)
y_delay = train_data['days_behind_schedule'].fillna(0)

delay_metrics = delay_predictor.train(X_delay, y_delay)
print(f"✅ Delay Model - Test RMSE: {delay_metrics['test_rmse']:.2f}, R²: {delay_metrics['test_r2']:.3f}")

# Cost Overrun Model
print("\n💰 Training Cost Overrun Predictor...")
cost_predictor = CostOverrunPredictor('xgboost')

X_cost = cost_predictor.prepare_features(train_data)
y_cost = train_data['budget_variance_percent'].fillna(0)

cost_metrics = cost_predictor.train(X_cost, y_cost)
print(f"✅ Cost Model - Test RMSE: {cost_metrics['test_rmse']:.2f}, R²: {cost_metrics['test_r2']:.3f}")

# Resource Bottleneck Detector
print("\n👥 Training Resource Bottleneck Detector...")
resource_detector = ResourceBottleneckDetector()

X_resource = resource_detector.prepare_features(processed_resources)
resource_detector.fit(X_resource)
print("✅ Resource Detector trained successfully")

print("\n🎉 All models trained successfully!")

## 7. Feature Importance Analysis

In [ ]:
# Get feature importance
delay_importance = delay_predictor.get_feature_importance()

# Plot
plt.figure(figsize=(10, 6))
plt.barh(delay_importance['feature'], delay_importance['importance'])
plt.xlabel('Importance Score')
plt.title('Feature Importance for Delay Prediction')
plt.tight_layout()
plt.show()

print("📊 Top 5 Most Important Features:")
print(delay_importance.head())

## 8. Make Predictions

In [ ]:
# Select a sample project for prediction
sample_project = train_data[train_data['project_id'] == 1].tail(1)

# Delay prediction
X_test_delay = delay_predictor.prepare_features(sample_project)
delay_pred, delay_conf = delay_predictor.predict_with_confidence(X_test_delay)

# Cost prediction
X_test_cost = cost_predictor.prepare_features(sample_project)
cost_pred = cost_predictor.predict(X_test_cost)

# Resource anomalies
sample_resources = processed_resources[processed_resources['project_id'] == 1]
bottlenecks = resource_detector.identify_bottlenecks(sample_resources)

print("🔮 Predictions for Project 1:")
print(f"\nPredicted Delay: {delay_pred[0]:.1f} days (Confidence: {delay_conf[0]:.2f})")
print(f"Predicted Cost Overrun: {cost_pred[0]:.1f}%")
print(f"Resource Bottlenecks Detected: {bottlenecks['is_bottleneck'].sum()}")

# Calculate overall risk score
calculator = RiskScoreCalculator()
risk_report = calculator.generate_risk_report({
    'predicted_delay_days': delay_pred[0],
    'delay_confidence': delay_conf[0],
    'predicted_cost_overrun_percent': cost_pred[0],
    'anomaly_score': bottlenecks['anomaly_score'].mean(),
    'avg_utilization_rate': bottlenecks['utilization_rate'].mean(),
    'bug_rate': 0.1,
    'reopen_rate': 0.15
})

print(f"\n⚠️  Overall Risk Score: {risk_report['overall_score']:.1f}/100")
print(f"🎯 Risk Level: {risk_report['risk_level'].upper()}")
print(f"\n📊 Risk Breakdown:")
for risk_type, score in risk_report['risk_scores'].items():
    print(f"  - {risk_type.replace('_', ' ').title()}: {score:.1f}")

## 9. Anomaly Detection

In [ ]:
# Visualize anomalies
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Anomaly scores
colors = ['red' if x else 'green' for x in bottlenecks['is_bottleneck']]
axes[0].scatter(bottlenecks['utilization_rate'], 
               bottlenecks['anomaly_score'],
               c=colors, alpha=0.6, s=100)
axes[0].set_title('Resource Anomaly Detection')
axes[0].set_xlabel('Utilization Rate')
axes[0].set_ylabel('Anomaly Score')
axes[0].axvline(x=1.0, color='orange', linestyle='--', label='100% Utilization')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Utilization distribution
bottlenecks['utilization_rate'].hist(bins=20, ax=axes[1], edgecolor='black')
axes[1].axvline(x=1.0, color='red', linestyle='--', label='Optimal')
axes[1].set_title('Utilization Rate Distribution')
axes[1].set_xlabel('Utilization Rate')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🚨 Anomalies Detected:")
print(f"  - Over-allocated resources: {bottlenecks['is_overallocated'].sum()}")
print(f"  - Under-utilized resources: {bottlenecks['is_underutilized'].sum()}")
print(f"  - Bottleneck resources: {bottlenecks['is_bottleneck'].sum()}")

## 10. Summary Report

In [ ]:
print("="*60)
print("     AI-POWERED RISK EARLY WARNING SYSTEM - SUMMARY")
print("="*60)

print("\n📊 SYSTEM CAPABILITIES:")
print("  ✅ Data Processing & Feature Engineering")
print("  ✅ Delay Risk Prediction (XGBoost)")
print("  ✅ Cost Overrun Forecasting")
print("  ✅ Resource Bottleneck Detection (Isolation Forest)")
print("  ✅ Comprehensive Risk Scoring")
print("  ✅ Anomaly Detection")

print("\n📈 MODEL PERFORMANCE:")
print(f"  Delay Predictor R²: {delay_metrics['test_r2']:.3f}")
print(f"  Cost Predictor R²: {cost_metrics['test_r2']:.3f}")
print(f"  Resource Detector: Trained on {len(processed_resources)} samples")

print("\n🎯 RISK ANALYSIS:")
print(f"  Projects Analyzed: {len(projects_df)}")
print(f"  Metrics Processed: {len(processed_metrics)}")
print(f"  Resources Monitored: {processed_resources['user_id'].nunique()}")

print("\n⚠️  KEY INSIGHTS:")
avg_velocity = processed_metrics['sprint_velocity'].mean()
avg_completion = processed_metrics['completion_rate'].mean()
avg_utilization = processed_resources['utilization_rate'].mean()

print(f"  Average Sprint Velocity: {avg_velocity:.1f} points/day")
print(f"  Average Completion Rate: {avg_completion:.1%}")
print(f"  Average Resource Utilization: {avg_utilization:.1%}")
print(f"  Projects Behind Schedule: {(processed_metrics['days_behind_schedule'] > 0).sum()}")

print("\n🚀 NEXT STEPS:")
print("  1. Start the FastAPI backend: uvicorn app.main:app --reload")
print("  2. Access API docs: http://localhost:8000/docs")
print("  3. View dashboard: http://localhost:3000/dashboard")
print("  4. Upload real data via /api/v1/upload endpoints")
print("  5. Retrain models periodically for better accuracy")

print("\n" + "="*60)
print("          System Ready for Production Use! 🎉")
print("="*60)